# Comparative Analysis: Attention U-Net vs TransUNet
**Skin Lesion Segmentation (ISIC Dataset)**

This notebook automatically generates our modular PyTorch codebase using `%%writefile` and then runs the entire training and evaluation pipeline end-to-end.

### Instructions:
1. Attach the ISIC 2018 dataset to your Kaggle notebook.
2. If your dataset folder isn't named exactly `isic-2018-challenge`, update the `DATA_ROOT` variable in the `config.py` cell below.
3. Click **Run All**!

In [ ]:
!pip install albumentations opencv-python-headless scikit-image tqdm scipy matplotlib


In [ ]:
%%writefile config.py
import os

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 10
MIN_LR = 1e-7
SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

DATA_ROOT = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation" 

IMAGE_DIR = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1-2_Training_Input"
MASK_DIR = "/kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation/ISIC2018_Task1_Training_GroundTruth"

OUTPUT_DIR = "./outputs"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

DEVICE = "cuda"


In [ ]:
%%writefile losses.py
import torch
import torch.nn as nn


class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCELoss()

    def forward(self, pred, target):
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)

        intersection = (pred_flat * target_flat).sum()
        dice_loss = 1 - (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )

        bce_loss = self.bce(pred_flat, target_flat)

        return dice_loss + bce_loss


In [ ]:
%%writefile metrics.py
import numpy as np
from scipy.ndimage import distance_transform_edt


def dice_score(pred, target, smooth=1.0):
    pred_flat = pred.flatten()
    target_flat = target.flatten()
    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)


def iou_score(pred, target, smooth=1.0):
    pred_flat = pred.flatten()
    target_flat = target.flatten()
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)


def hausdorff_distance(pred, target):
    """Directed Hausdorff distance between binary masks."""
    pred = pred.astype(bool)
    target = target.astype(bool)

    if not pred.any() and not target.any():
        return 0.0
    if not pred.any() or not target.any():
        return float("inf")

    dt_pred = distance_transform_edt(~pred)
    dt_target = distance_transform_edt(~target)

    hd_pred_to_target = dt_target[pred].max()
    hd_target_to_pred = dt_pred[target].max()

    return max(hd_pred_to_target, hd_target_to_pred)


In [ ]:
%%writefile dataset.py
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import config


def get_train_transforms():
    return A.Compose([
        A.Resize(config.IMAGE_SIZE, config.IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(config.IMAGE_SIZE, config.IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        mask = mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.tensor(mask).unsqueeze(0)
        return img, mask.float()


def get_loaders():
    images = sorted([
        os.path.join(config.IMAGE_DIR, f)
        for f in os.listdir(config.IMAGE_DIR)
        if f.endswith((".jpg", ".png"))
    ])

    masks = []
    for img_path in images:
        name = os.path.splitext(os.path.basename(img_path))[0]
        mask_name = name + "_segmentation.png"
        mask_path = os.path.join(config.MASK_DIR, mask_name)
        masks.append(mask_path)

    valid_pairs = [(i, m) for i, m in zip(images, masks) if os.path.exists(m)]
    images, masks = zip(*valid_pairs)
    images, masks = list(images), list(masks)

    train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
        images, masks, test_size=(1 - config.TRAIN_RATIO), random_state=config.SEED
    )
    relative_val = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)
    val_imgs, test_imgs, val_masks, test_masks = train_test_split(
        temp_imgs, temp_masks, test_size=(1 - relative_val), random_state=config.SEED
    )

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    train_ds = ISICDataset(train_imgs, train_masks, transform=get_train_transforms())
    val_ds = ISICDataset(val_imgs, val_masks, transform=get_val_transforms())
    test_ds = ISICDataset(test_imgs, test_masks, transform=get_val_transforms())

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)

    return train_loader, val_loader, test_loader, test_ds


In [ ]:
%%writefile attention_unet.py
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(g_ch, inter_ch, 1, bias=False),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(x_ch, inter_ch, 1, bias=False),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)

        self.enc1 = ConvBlock(in_channels, features[0])
        self.enc2 = ConvBlock(features[0], features[1])
        self.enc3 = ConvBlock(features[1], features[2])
        self.enc4 = ConvBlock(features[2], features[3])

        self.bottleneck = ConvBlock(features[3], features[3] * 2)

        self.up4 = nn.ConvTranspose2d(features[3] * 2, features[3], 2, stride=2)
        self.att4 = AttentionGate(features[3], features[3], features[3] // 2)
        self.dec4 = ConvBlock(features[3] * 2, features[3])

        self.up3 = nn.ConvTranspose2d(features[3], features[2], 2, stride=2)
        self.att3 = AttentionGate(features[2], features[2], features[2] // 2)
        self.dec3 = ConvBlock(features[2] * 2, features[2])

        self.up2 = nn.ConvTranspose2d(features[2], features[1], 2, stride=2)
        self.att2 = AttentionGate(features[1], features[1], features[1] // 2)
        self.dec2 = ConvBlock(features[1] * 2, features[1])

        self.up1 = nn.ConvTranspose2d(features[1], features[0], 2, stride=2)
        self.att1 = AttentionGate(features[0], features[0], features[0] // 2)
        self.dec1 = ConvBlock(features[0] * 2, features[0])

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        e4 = self.att4(d4, e4)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        e3 = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        e2 = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        e1 = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))


In [ ]:
%%writefile transunet.py
import torch
import torch.nn as nn
import math


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block for CNN encoder."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h)[0]
        x = x + self.mlp(self.norm2(x))
        return x


class TransUNet(nn.Module):
    def __init__(
        self,
        img_size=256,
        in_channels=3,
        out_channels=1,
        embed_dim=512,
        num_heads=8,
        num_layers=6,
        encoder_channels=[64, 128, 256],
        dropout=0.1,
    ):
        super().__init__()
        self.img_size = img_size

        # CNN encoder — downsample by 2 at each stage
        self.enc1 = ResBlock(in_channels, encoder_channels[0], stride=1)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ResBlock(encoder_channels[0], encoder_channels[1], stride=1)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ResBlock(encoder_channels[1], encoder_channels[2], stride=1)
        self.pool3 = nn.MaxPool2d(2)

        # after 3 pools: 256 -> 32
        self.enc_bridge = ResBlock(encoder_channels[2], encoder_channels[2], stride=1)
        self.pool_bridge = nn.MaxPool2d(2)
        # now 16x16

        feat_size = img_size // 16
        num_patches = feat_size * feat_size

        self.patch_embed = nn.Conv2d(encoder_channels[2], embed_dim, 1)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        self.transformer = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads, dropout=dropout) for _ in range(num_layers)]
        )

        self.proj_back = nn.Conv2d(embed_dim, encoder_channels[2], 1)

        # decoder — upsample keeps channels matching the skip connection
        self.up3 = nn.ConvTranspose2d(encoder_channels[2], encoder_channels[2], 2, stride=2)
        self.dec3 = ConvBlock(encoder_channels[2] * 2, encoder_channels[2])

        self.up2 = nn.ConvTranspose2d(encoder_channels[2], encoder_channels[2], 2, stride=2)
        self.dec2 = ConvBlock(encoder_channels[2] + encoder_channels[2], encoder_channels[1])

        self.up1 = nn.ConvTranspose2d(encoder_channels[1], encoder_channels[1], 2, stride=2)
        self.dec1 = ConvBlock(encoder_channels[1] + encoder_channels[1], encoder_channels[0])

        self.up0 = nn.ConvTranspose2d(encoder_channels[0], encoder_channels[0], 2, stride=2)
        self.dec0 = ConvBlock(encoder_channels[0], encoder_channels[0])

        self.final = nn.Conv2d(encoder_channels[0], out_channels, 1)

        self._init_pos_embed(num_patches, embed_dim)

    def _init_pos_embed(self, num_patches, embed_dim):
        pos = torch.arange(num_patches).unsqueeze(1).float()
        dim = torch.arange(embed_dim).unsqueeze(0).float()
        angles = pos / (10000 ** (2 * (dim // 2) / embed_dim))
        pe = torch.zeros(1, num_patches, embed_dim)
        pe[0, :, 0::2] = torch.sin(angles[:, 0::2])
        pe[0, :, 1::2] = torch.cos(angles[:, 1::2])
        self.pos_embed.data.copy_(pe)

    def forward(self, x):
        # encoder
        e1 = self.enc1(x)          # B, 64, 256, 256
        e2 = self.enc2(self.pool1(e1))  # B, 128, 128, 128
        e3 = self.enc3(self.pool2(e2))  # B, 256, 64, 64
        e_bridge = self.enc_bridge(self.pool3(e3))  # B, 256, 32, 32
        e4 = self.pool_bridge(e_bridge)  # B, 256, 16, 16

        # patch embed + transformer
        B, C, H, W = e4.shape
        t = self.patch_embed(e4)        # B, embed_dim, 16, 16
        t = t.flatten(2).transpose(1, 2)  # B, 256, embed_dim
        t = self.pos_drop(t + self.pos_embed)
        t = self.transformer(t)
        t = t.transpose(1, 2).view(B, -1, H, W)
        t = self.proj_back(t)           # B, 256, 16, 16

        # decoder with skip connections
        d3 = self.up3(t)                        # B, 256, 32, 32
        d3 = self.dec3(torch.cat([d3, e_bridge], dim=1))

        d2 = self.up2(d3)                       # B, 128, 64, 64
        d2 = self.dec2(torch.cat([d2, e3], dim=1))

        d1 = self.up1(d2)                       # B, 64, 128, 128
        d1 = self.dec1(torch.cat([d1, e2], dim=1))

        d0 = self.up0(d1)                       # B, 64, 256, 256
        d0 = self.dec0(d0)

        return torch.sigmoid(self.final(d0))


In [ ]:
%%writefile train.py
import time
import csv
import copy
import torch
from tqdm import tqdm
import config
from losses import DiceBCELoss
from metrics import dice_score


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0

    for images, masks in tqdm(loader, desc="Train", leave=False):
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        pred_np = (preds > 0.5).float()
        for i in range(images.size(0)):
            running_dice += dice_score(
                pred_np[i, 0].cpu().numpy(),
                masks[i, 0].cpu().numpy(),
            )

    n = len(loader.dataset)
    return running_loss / n, running_dice / n


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0

    for images, masks in tqdm(loader, desc="Val", leave=False):
        images, masks = images.to(device), masks.to(device)
        preds = model(images)
        loss = criterion(preds, masks)

        running_loss += loss.item() * images.size(0)
        pred_np = (preds > 0.5).float()
        for i in range(images.size(0)):
            running_dice += dice_score(
                pred_np[i, 0].cpu().numpy(),
                masks[i, 0].cpu().numpy(),
            )

    n = len(loader.dataset)
    return running_loss / n, running_dice / n


def train_model(model, train_loader, val_loader, model_name="model"):
    device = torch.device(config.DEVICE)
    model = model.to(device)

    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5, min_lr=config.MIN_LR)

    best_dice = 0.0
    patience_counter = 0
    best_weights = None
    history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": [], "lr": []}

    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")

    start_time = time.time()

    for epoch in range(config.EPOCHS):
        train_loss, train_dice = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_dice = validate(model, val_loader, criterion, device)

        scheduler.step(val_dice)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_dice"].append(train_dice)
        history["val_dice"].append(val_dice)

        lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(lr)
        print(
            f"Epoch {epoch+1}/{config.EPOCHS} | "
            f"Train Loss: {train_loss:.4f} Dice: {train_dice:.4f} | "
            f"Val Loss: {val_loss:.4f} Dice: {val_dice:.4f} | "
            f"LR: {lr:.2e}"
        )

        if val_dice > best_dice:
            best_dice = val_dice
            patience_counter = 0
            best_weights = copy.deepcopy(model.state_dict())
            ckpt_path = f"{config.CHECKPOINT_DIR}/{model_name}_best.pth"
            torch.save(best_weights, ckpt_path)
            print(f"  -> Saved best model (Dice: {best_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                break

    train_time = time.time() - start_time
    model.load_state_dict(best_weights)
    print(f"Training time: {train_time:.1f}s | Best Val Dice: {best_dice:.4f}")

    csv_path = f"{config.OUTPUT_DIR}/{model_name}_training_log.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "train_dice", "val_dice", "lr"])
        for i in range(len(history["train_loss"])):
            writer.writerow([
                i + 1,
                f"{history['train_loss'][i]:.6f}",
                f"{history['val_loss'][i]:.6f}",
                f"{history['train_dice'][i]:.6f}",
                f"{history['val_dice'][i]:.6f}",
                f"{history['lr'][i]:.2e}",
            ])
    print(f"Saved training log: {csv_path}")

    return model, history, train_time


In [ ]:
%%writefile evaluate.py
import time
import numpy as np
import torch
from tqdm import tqdm
import config
from metrics import dice_score, iou_score, hausdorff_distance


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


@torch.no_grad()
def evaluate_model(model, test_loader, model_name="model"):
    device = torch.device(config.DEVICE)
    model = model.to(device)
    model.eval()

    dice_scores = []
    iou_scores = []
    hd_scores = []
    inference_times = []

    for images, masks in tqdm(test_loader, desc=f"Evaluating {model_name}"):
        images, masks = images.to(device), masks.to(device)

        start = time.time()
        preds = model(images)
        torch.cuda.synchronize() if device.type == "cuda" else None
        inference_times.append(time.time() - start)

        preds_bin = (preds > 0.5).float()

        for i in range(images.size(0)):
            p = preds_bin[i, 0].cpu().numpy()
            t = masks[i, 0].cpu().numpy()

            dice_scores.append(dice_score(p, t))
            iou_scores.append(iou_score(p, t))

            hd = hausdorff_distance(p, t)
            if hd != float("inf"):
                hd_scores.append(hd)

    results = {
        "model": model_name,
        "params": count_parameters(model),
        "dice_mean": np.mean(dice_scores),
        "dice_std": np.std(dice_scores),
        "iou_mean": np.mean(iou_scores),
        "iou_std": np.std(iou_scores),
        "hd_mean": np.mean(hd_scores) if hd_scores else float("nan"),
        "hd_std": np.std(hd_scores) if hd_scores else float("nan"),
        "avg_inference_time": np.mean(inference_times),
    }

    return results, dice_scores, iou_scores


def print_comparison(results_list):
    print(f"\n{'='*70}")
    print(f"{'METRIC':<25} | {'Attention U-Net':>18} | {'TransUNet':>18}")
    print(f"{'-'*70}")

    r1, r2 = results_list[0], results_list[1]

    print(f"{'Parameters':<25} | {r1['params']:>18,} | {r2['params']:>18,}")
    print(f"{'Dice Score':<25} | {r1['dice_mean']:.4f} ± {r1['dice_std']:.4f}  | {r2['dice_mean']:.4f} ± {r2['dice_std']:.4f}")
    print(f"{'IoU Score':<25} | {r1['iou_mean']:.4f} ± {r1['iou_std']:.4f}  | {r2['iou_mean']:.4f} ± {r2['iou_std']:.4f}")
    print(f"{'Hausdorff Distance':<25} | {r1['hd_mean']:.2f} ± {r1['hd_std']:.2f}    | {r2['hd_mean']:.2f} ± {r2['hd_std']:.2f}")
    print(f"{'Avg Inference Time (s)':<25} | {r1['avg_inference_time']:.4f}            | {r2['avg_inference_time']:.4f}")
    print(f"{'='*70}\n")


In [ ]:
%%writefile visualize.py
import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
import config


MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])


def denormalize(img_tensor):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img, 0, 1)


@torch.no_grad()
def plot_predictions(model, dataset, model_name, n=8, save=True):
    device = torch.device(config.DEVICE)
    model = model.to(device)
    model.eval()

    indices = np.random.choice(len(dataset), min(n, len(dataset)), replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))

    for row, idx in enumerate(indices):
        img, mask = dataset[idx]
        pred = model(img.unsqueeze(0).to(device))
        pred = (pred > 0.5).float()

        axes[row, 0].imshow(denormalize(img))
        axes[row, 0].set_title("Input Image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask[0].cpu().numpy(), cmap="gray")
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred[0, 0].cpu().numpy(), cmap="gray")
        axes[row, 2].set_title(f"{model_name} Prediction")
        axes[row, 2].axis("off")

    plt.suptitle(f"Qualitative Results — {model_name}", fontsize=16, y=1.01)
    plt.tight_layout()

    if save:
        path = f"{config.FIGURE_DIR}/{model_name}_predictions.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()


def plot_training_curves(history1, history2, name1="Attention U-Net", name2="TransUNet", save=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history1["train_loss"], label=f"{name1} Train")
    axes[0].plot(history1["val_loss"], label=f"{name1} Val", linestyle="--")
    axes[0].plot(history2["train_loss"], label=f"{name2} Train")
    axes[0].plot(history2["val_loss"], label=f"{name2} Val", linestyle="--")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history1["train_dice"], label=f"{name1} Train")
    axes[1].plot(history1["val_dice"], label=f"{name1} Val", linestyle="--")
    axes[1].plot(history2["train_dice"], label=f"{name2} Train")
    axes[1].plot(history2["val_dice"], label=f"{name2} Val", linestyle="--")
    axes[1].set_title("Dice Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save:
        path = f"{config.FIGURE_DIR}/training_curves.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()


def load_history_from_csv(csv_path):
    history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": [], "lr": []}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            history["train_loss"].append(float(row["train_loss"]))
            history["val_loss"].append(float(row["val_loss"]))
            history["train_dice"].append(float(row["train_dice"]))
            history["val_dice"].append(float(row["val_dice"]))
            history["lr"].append(float(row["lr"]))
    return history


def plot_training_curves_from_csv(csv1, csv2, name1="Attention U-Net", name2="TransUNet", save=True):
    h1 = load_history_from_csv(csv1)
    h2 = load_history_from_csv(csv2)
    plot_training_curves(h1, h2, name1, name2, save)


In [ ]:
%%writefile main.py
import torch
import numpy as np
import random
import config
from dataset import get_loaders
from attention_unet import AttentionUNet
from transunet import TransUNet
from train import train_model
from evaluate import evaluate_model, print_comparison
from visualize import plot_predictions, plot_training_curves


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def main():
    set_seed(config.SEED)
    train_loader, val_loader, test_loader, test_ds = get_loaders()

    # --- Attention U-Net ---
    att_unet = AttentionUNet(in_channels=3, out_channels=1)
    att_unet, hist_att, time_att = train_model(att_unet, train_loader, val_loader, model_name="AttentionUNet")

    # --- TransUNet ---
    trans_unet = TransUNet(img_size=config.IMAGE_SIZE, in_channels=3, out_channels=1)
    trans_unet, hist_trans, time_trans = train_model(trans_unet, train_loader, val_loader, model_name="TransUNet")

    # --- Evaluation ---
    results_att, _, _ = evaluate_model(att_unet, test_loader, model_name="Attention U-Net")
    results_trans, _, _ = evaluate_model(trans_unet, test_loader, model_name="TransUNet")

    results_att["train_time"] = time_att
    results_trans["train_time"] = time_trans

    print_comparison([results_att, results_trans])

    # --- Visualizations ---
    plot_predictions(att_unet, test_ds, "Attention_UNet", n=8)
    plot_predictions(trans_unet, test_ds, "TransUNet", n=8)
    plot_training_curves(hist_att, hist_trans)

    print("Done!")


if __name__ == "__main__":
    main()


## Run the Complete Pipeline
This will train both models, evaluate them, print the metrics, and save figures to the `outputs/figures` folder in your Kaggle working directory.

In [ ]:
!python main.py
